## 00 · Input audio

Generates the spoken questions with **Kokoro-82M** (Apache-2.0, ONNX build, CPU) for dataset **v4** (the default) or **v2**. Kokoro is a different TTS model from every candidate's (they speak with Piper), so the ASR stage isn't tested on its own family of voices.

v2 reads the golden questions as written. v4 rewords 5 of them the way callers talk, with fillers, a restart and a corrected order number (`data/wording_v4.jsonl`); the expected tool calls and facts don't change. Each golden-set speaker gets a distinct US or UK English voice, with light background noise (20 dB SNR). Clips go to `input/<version>/` with a `manifest.json` of voices; earlier versions stay where they are, because earlier traces reference them by path.

Listen to every clip at the end once and fix any line the voice misread. That listen is the human check on the ground truth. **Illustrative data.**

In [ ]:
%pip install -q kokoro-onnx
dbutils.library.restartPython()

In [ ]:
import json, os, sys, time

sys.path.insert(0, os.path.abspath(".."))
from voice_evals import config as C
from voice_evals.audio import add_noise, download_file, wav_bytes, write_bytes

dbutils.widgets.text("dataset_version", C.DATASET_VERSION, "Dataset version (v2 or v4)")
VERSION = dbutils.widgets.get("dataset_version").strip() or C.DATASET_VERSION
assert VERSION in ("v2", "v4"), "v1 comes from scripts/make_input_audio.py, v3 from 00_noisy_audio"
OUT = C.INPUT_DIRS[VERSION]
os.makedirs(OUT, exist_ok=True)
for name, url in C.KOKORO_FILES.items():
    path = download_file(C.kokoro_path(name), url)
    print(name, round(os.path.getsize(path) / 1e6, 1), "MB")
golden = C.load_golden(VERSION)  # v4 rewords some questions
print(f"{len(golden)} questions -> {OUT}")

In [ ]:
from kokoro_onnx import Kokoro

kokoro = Kokoro(C.kokoro_path("kokoro-v1.0.onnx"), C.kokoro_path("voices-v1.0.bin"))
manifest, rows = {}, []
for i, g in enumerate(golden):
    voice, lang = C.KOKORO_VOICES[g["voice"]]
    start = time.perf_counter()
    samples, rate = kokoro.create(g["spoken_text"], voice=voice, speed=1.0, lang=lang)
    path = f"{OUT}/{g['utterance_id']}.wav"
    write_bytes(path, wav_bytes(add_noise(samples, C.INPUT_SNR_DB, seed=i), rate))
    manifest[g["utterance_id"]] = {"voice": f"Kokoro {voice}", "lang": lang, "engine": "Kokoro-82M (ONNX)",
                                   "sample_rate_hz": rate, "snr_db": C.INPUT_SNR_DB}
    rows.append((g["utterance_id"], voice, lang, round(len(samples) / rate, 1), round(time.perf_counter() - start, 1), g["spoken_text"]))
write_bytes(f"{OUT}/manifest.json", json.dumps(manifest, indent=1).encode())
display(spark.createDataFrame(rows, "utterance_id STRING, voice STRING, lang STRING, seconds DOUBLE, synth_s DOUBLE, spoken_text STRING"))

In [ ]:
from IPython.display import Audio, Markdown, display as show

for g in golden:
    show(Markdown(f"**{g['utterance_id']}** · {manifest[g['utterance_id']]['voice']} · {g['spoken_text']}"),
         Audio(filename=f"{OUT}/{g['utterance_id']}.wav"))

In [ ]:
dbutils.notebook.exit(json.dumps({"input_dir": OUT, "clips": len(rows),
                                  "seconds": [r[3] for r in rows], "voices": sorted({r[1] for r in rows})}))